# Chapter 16 &mdash; Why Converting CNF to DNF Is Not a Free Lunch

**Concept 15 of the Chapter 16 decomposition:** *Why Converting CNF to DNF Is Not a Free Lunch*

DNF satisfiability is linear. So convert the CNF and win? The conversion is exponential, and that is the whole answer.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-CNF-To-DNF-Is-No-Free-Lunch/Concept-CNF-To-DNF-Is-No-Free-Lunch.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Bdd            import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Satisfiability of a **DNF** formula is trivial. Scan each product term; accept unless it
contains some variable together with its negation. **Linear time.**

So why is SAT hard? Because SAT is posed over **CNF**, and distributing to get from one
to the other multiplies out: $k$ clauses of $w$ literals become $w^k$ products.

$$(a\vee b\vee c\vee d)\wedge(e\vee f\vee g\vee h)\wedge\cdots
\quad\longrightarrow\quad 4^8 = 65536 \text{ terms}$$

A linear-time algorithm run on an input exponentially larger than the one you were
handed has bought you nothing. Every student is tempted by this once; it is worth
being tempted carefully, and then measuring.

Measuring shows something sharper than the arithmetic. The **diagram** for such a
family grows *linearly* while its **paths** grow exponentially &mdash; so the BDD
stores that 65536-term DNF compactly, and still cannot enumerate it any faster.

**Input encoding is part of the complexity claim, not a footnote to it.**

## 2. Definitions

### A family of CNFs, multiplied out

In [ ]:
def wide_cnf(k, w=4):
    # k clauses of w literals, all variables distinct
    names = [['x%d_%d' % (i, j) for j in range(w)] for i in range(k)]
    flat  = [v for row in names for v in row]
    body  = ['c%d = %s' % (i, ' | '.join(row)) for i, row in enumerate(names)]
    return ('Var_Order : ' + ' '.join(flat) + '\n'
            + '\n'.join(body) + '\n'
            + 'Main_Exp : ' + ' & '.join('c%d' % i for i in range(k)))

print(wide_cnf(2))

<!-- nav-strip -->

---

&larr;&nbsp;[Ch16&nbsp;14.&nbsp;Both Normal Forms, Read Off One Diagram](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-Normal-Forms-From-BDD/Concept-Normal-Forms-From-BDD.ipynb) &nbsp;&middot;&nbsp; [**Chapter 16** index](https://github.com/ganeshutah/Jove/blob/master/Chapter16/README.md) &nbsp;&middot;&nbsp; [Ch16&nbsp;16.&nbsp;Counting Solutions: a BDD Does #SAT in One Pass](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-Counting-Solutions-Sharp-SAT/Concept-Counting-Solutions-Sharp-SAT.ipynb)&nbsp;&rarr;

---

## 3. Tests

Grow $k$ and watch the two columns part company.

In [ ]:
print('%3s %6s %10s %12s %8s' % ('k', 'vars', 'w^k terms', 'BDD paths', 'nodes'))
for k in range(1, 6):
    b = wide_cnf(k)
    d = bdd(b)
    print('%3d %6d %10d %12d %8d'
          % (k, 4 * k, 4 ** k, len(paths(d, 1)), d.nodes))
print()
print('The DNF term count is 4^k exactly -- that is the distribution law.')
print('The DIAGRAM grows by 4 nodes per clause.  Linear structure, an')
print('exponential number of paths through it.')

**Look at one.** Three clauses: 14 nodes, and 64 paths through them. The picture is the argument &mdash; the structure is small, the number of ways through it is not.

In [ ]:
bdd(wide_cnf(3))

So the blow-up is real, and it is in the **answer**, not in the method.

In [ ]:
d = bdd(wide_cnf(5))
assert len(paths(d, 1)) == 4 ** 5
print('k=5:  %d nodes hold %d product terms.'
      % (d.nodes, len(paths(d, 1))))
print()
print('Storing the DNF compactly is not the same as having it.  To USE it')
print('for the linear-time DNF-SAT test you must walk the terms, and there')
print('are 4^k of them however they are stored.')

**And the test itself is honest.** DNF-SAT really is linear &mdash; on a DNF. Here it is, so the claim is not taken on trust.

In [ ]:
def dnf_sat(terms):
    # a product term is satisfiable unless it holds x and ~x
    for t in terms:
        lits = set(t)
        if not any(('~' + l) in lits for l in lits if not l.startswith('~')):
            return True
    return False

ps = paths(bdd(wide_cnf(3)), 1)
print('DNF-SAT over %d terms: %s   (one scan, no backtracking)'
      % (len(ps), dnf_sat(ps)))
print()
print('Linear in the DNF.  The DNF was exponential in the CNF you started')
print('from.  That is the whole of the answer.')

## 4. Exercises


1. Redo the table with clauses of **two** literals instead of four. The growth is
   still exponential &mdash; what changed, and what did not?
2. `wide_cnf` uses all-distinct variables. Make the clauses share variables and
   re-measure. Why does sharing shrink the path count, and does it shrink it enough
   to matter?
3. Concept 12 introduced **equisatisfiable** Tseitin encoding, which is *linear*.
   Why is that not the same cheat in reverse?
4. State precisely what would have to be true of the CNF-to-DNF conversion for the
   cheat to work, and say which known result it would contradict.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter16/Concept-CNF-To-DNF-Is-No-Free-Lunch')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')